# AME 5052 – Applied Machine Learning Lab
## Assignment 2: Clustering, Ensemble Learning and Dimensionality Reduction

**Topics:** Distance and similarity, K-Means, Hierarchical clustering, dendrograms and linkage methods, Random Forest, Boosting, Gaussian Mixture Models (GMM), and PCA.

This notebook uses standard scikit-learn datasets and generated data so it runs independently of external files.

## 2(a). Computing Distances and Similarities; K-Means and Hierarchical Clustering

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.metrics import pairwise_distances
from scipy.cluster.hierarchy import linkage, dendrogram

iris = load_iris(as_frame=True)
X = iris.data
y_true = iris.target
X_scaled = StandardScaler().fit_transform(X)

# Euclidean and Manhattan distances between the first two observations
print("Euclidean distance:", pairwise_distances(X_scaled[:2], metric="euclidean")[0, 1])
print("Manhattan distance:", pairwise_distances(X_scaled[:2], metric="manhattan")[0, 1])

# Cosine similarity = 1 - cosine distance
cosine_distance = pairwise_distances(X_scaled[:2], metric="cosine")[0, 1]
print("Cosine similarity:", 1 - cosine_distance)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
print("K-Means silhouette score:", round(silhouette_score(X_scaled, kmeans_labels), 4))
print("K-Means adjusted Rand index:", round(adjusted_rand_score(y_true, kmeans_labels), 4))

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=kmeans_labels, cmap="viridis", s=35)
plt.xlabel("Standardized feature 1")
plt.ylabel("Standardized feature 2")
plt.title("K-Means Clustering (first two features)")
plt.show()

### Hierarchical clustering and linkage methods

In [ ]:
linkage_methods = ["single", "complete", "average", "ward"]
hierarchical_results = []

for method in linkage_methods:
    model = AgglomerativeClustering(n_clusters=3, linkage=method)
    labels = model.fit_predict(X_scaled)
    hierarchical_results.append({
        "Linkage": method,
        "Silhouette Score": silhouette_score(X_scaled, labels),
        "Adjusted Rand Index": adjusted_rand_score(y_true, labels)
    })

display(pd.DataFrame(hierarchical_results).round(4))

In [ ]:
Z = linkage(X_scaled, method="ward")
plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode="lastp", p=30)
plt.title("Hierarchical Clustering Dendrogram (Ward linkage)")
plt.xlabel("Cluster / observation")
plt.ylabel("Distance")
plt.show()

**Analysis:** Euclidean distance measures straight-line distance; Manhattan distance sums absolute coordinate differences; cosine similarity measures the angle between vectors. K-Means partitions observations around centroids. Hierarchical clustering builds nested clusters. Single, complete, average and Ward linkage differ in how they measure the distance between clusters.

## 2(b). Ensemble Methods: Bagging (Random Forest) and Boosting

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y_true, test_size=0.2, random_state=42, stratify=y_true
)

ensemble_models = {
    "Random Forest (Bagging)": RandomForestClassifier(n_estimators=100, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

ensemble_results = []
for name, model in ensemble_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    ensemble_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision (macro)": precision_score(y_test, pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, pred, average="macro", zero_division=0),
        "F1 (macro)": f1_score(y_test, pred, average="macro", zero_division=0)
    })
    print("\n", name)
    print(classification_report(y_test, pred, target_names=iris.target_names))

display(pd.DataFrame(ensemble_results).round(4))

**Analysis:** Bagging trains multiple models on bootstrap samples and combines their predictions. Random Forest also uses random subsets of features when splitting. Boosting trains models sequentially, with later learners focusing on errors made by earlier learners. AdaBoost and Gradient Boosting are examples of boosting.

## 2(c). Gaussian Mixture Model (GMM) for Clustering

In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=3, covariance_type="full", random_state=42)
gmm_labels = gmm.fit_predict(X_scaled)

print("GMM silhouette score:", round(silhouette_score(X_scaled, gmm_labels), 4))
print("GMM adjusted Rand index:", round(adjusted_rand_score(y_true, gmm_labels), 4))
print("GMM converged:", gmm.converged_)
print("Number of iterations:", gmm.n_iter_)

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=gmm_labels, cmap="plasma", s=35)
plt.xlabel("Standardized feature 1")
plt.ylabel("Standardized feature 2")
plt.title("Gaussian Mixture Model Clustering")
plt.show()

**Analysis:** GMM is a probabilistic clustering model. It assumes data is generated from a mixture of Gaussian distributions. Unlike K-Means, GMM can model clusters with different shapes and provides probabilistic membership.

## 2(d). Principal Component Analysis (PCA) and Visualization

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total variance explained by 2 components:", round(pca.explained_variance_ratio_.sum(), 4))

plt.figure(figsize=(8, 5))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, cmap="viridis", s=35)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Visualization of Iris Dataset")
plt.legend(handles=scatter.legend_elements()[0], labels=list(iris.target_names), title="Class")
plt.grid(True)
plt.show()

**Conclusion:** PCA transforms correlated features into a smaller set of uncorrelated principal components while retaining as much variance as possible. The first two components are plotted to visualize the dataset in two dimensions. Clustering quality is assessed with silhouette score and adjusted Rand index; ensemble classifiers are assessed with accuracy, precision, recall and F1-score.